In [ ]:
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    backfill_skew_phase_fits,
)
from md_Helpers.voxel_fit import PHASE_FIT_METHOD_VERSION


# ================================================================
# Configuration
# ================================================================

APPLY = True             # False: preview; True: write validated results
LIMIT = None               # Canary batch size; use None for all remaining runs
RUN_IDS = None            # Example: ["20260915184126"]; None selects all
RETRY_FAILED = True      # True retries runs previously recorded as failed
MAX_ATTEMPTS = 3


# ================================================================
# Database and automatic backup
# ================================================================

paths = ProjectPaths()
database = SQLiteRunDatabase(paths.database)
database.initialize()

print(f"Database: {paths.database}")
print(f"Simulation root: {paths.top_directory}")
print(f"Model version: {PHASE_FIT_METHOD_VERSION}")
print(f"Mode: {'APPLY' if APPLY else 'PREVIEW'}")

if APPLY:
    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )

    backup_path = (
        paths.top_directory
        / "backups"
        / f"mdsims-before-skew-{timestamp}.sqlite3"
    )

    database.backup(backup_path)
    print(f"Database backup created: {backup_path}")


# ================================================================
# Preview or execute the resumable backfill
# ================================================================

results = backfill_skew_phase_fits(
    database,
    project_paths=paths,
    run_ids=RUN_IDS,
    limit=LIMIT,
    dry_run=not APPLY,
    retry_failed=RETRY_FAILED,
    max_attempts=MAX_ATTEMPTS,
)

display(results)

if results.empty:
    print("No eligible phase-separated runs were found.")
else:
    print("\nBatch status:")
    display(
        results["Status"]
        .value_counts()
        .rename_axis("Status")
        .reset_index(name="Number of runs")
    )

    failed = results[
        results["Status"].isin(
            [
                "Failed",
                "Failed_Not_Retried",
                "Max_Attempts",
            ]
        )
    ]

    if not failed.empty:
        print("\nRuns requiring attention:")
        display(failed)


# ================================================================
# Current migration history
# ================================================================

history = pd.DataFrame(
    database.query_phase_fit_history(
        method_version=PHASE_FIT_METHOD_VERSION,
    )
)

if not history.empty:
    print("\nComplete migration history summary:")

    display(
        history.groupby(
            ["Sim_Table", "Status"],
            dropna=False,
        )
        .size()
        .rename("Number of records")
        .reset_index()
    )

    display(
        history[
            [
                "Run_ID",
                "Sim_Table",
                "Status",
                "Attempt_Count",
                "N_Bins",
                "Liquid_Shape_Alpha",
                "rho_liquid",
                "rho_gas",
                "Error_Message",
            ]
        ].sort_values(
            ["Status", "Run_ID", "Sim_Table"]
        )
    )

Database: /exp/e961/data/MDsims-data/pnichols/SQL/mdsims.sqlite3
Simulation root: /exp/e961/data/MDsims-data/pnichols/SQL
Model version: terminal_5_saved_frames_skew_liquid_v1
Mode: APPLY
Database backup created: /exp/e961/data/MDsims-data/pnichols/SQL/backups/mdsims-before-skew-20261006T171400_078324Z.sqlite3
